In [5]:
from dataclasses import dataclass

from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from IPython.display import Markdown, display
from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False  
GRPC_HOST = "localhost:50052"

### Message type

In [6]:
@dataclass
class Message:
    content: str

### gRPC host

In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address=GRPC_HOST)
host.start()

### Routed agents (style, security, lead)

In [8]:
class StyleReviewer(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[],
            system_message=(
                "You review code for readability, naming, structure, and maintainability. "
                "Short bullets; mark info / warn / important. No security topics."
            ),
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class SecurityReviewer(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[],
            system_message=(
                "You review code for security: injection, secrets, unsafe IO, weak crypto, "
                "command execution, auth issues, etc. Short bullets; severity low/medium/high. No style topics."
            ),
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class LeadReviewer(RoutedAgent):
    def __init__(self, name: str, language: str = "python") -> None:
        super().__init__(name)
        self._language = language
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[],
            system_message=(
                "You synthesize specialist reviews only—do not invent new issues. "
                "Output: (1) executive summary, (2) merged findings by theme, "
                "(3) one recommendation: approve with nits / request changes / block."
            ),
        )

    def _wrap_code(self, code: str) -> str:
        return f"```{self._language}\n{code.strip()}\n```"

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        code = message.content.strip()
        block = self._wrap_code(code)
        style_prompt = "Review for STYLE and maintainability only.\n\n" + block
        security_prompt = "Review for SECURITY only.\n\n" + block
        r_style = await self.send_message(Message(content=style_prompt), AgentId("style_reviewer", "default"))
        r_sec = await self.send_message(Message(content=security_prompt), AgentId("security_reviewer", "default"))
        bundle = f"## Style review\n{r_style.content}\n\n## Security review\n{r_sec.content}\n"
        merge_prompt = "Specialist reviews follow. Synthesize per your system message.\n\n" + bundle
        tm = TextMessage(content=merge_prompt, source="user")
        synth = await self._delegate.on_messages([tm], ctx.cancellation_token)
        return Message(content=bundle + "\n## Lead summary\n\n" + synth.chat_message.content)

### Register workers

In [9]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:
    worker = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker.start()
    await StyleReviewer.register(worker, "style_reviewer", lambda: StyleReviewer("style_reviewer"))
    await SecurityReviewer.register(worker, "security_reviewer", lambda: SecurityReviewer("security_reviewer"))
    await LeadReviewer.register(worker, "lead_reviewer", lambda: LeadReviewer("lead_reviewer"))
    agent_id = AgentId("lead_reviewer", "default")
else:
    w_style = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await w_style.start()
    await StyleReviewer.register(w_style, "style_reviewer", lambda: StyleReviewer("style_reviewer"))

    w_sec = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await w_sec.start()
    await SecurityReviewer.register(w_sec, "security_reviewer", lambda: SecurityReviewer("security_reviewer"))

    worker = GrpcWorkerAgentRuntime(host_address=GRPC_HOST)
    await worker.start()
    await LeadReviewer.register(worker, "lead_reviewer", lambda: LeadReviewer("lead_reviewer"))
    agent_id = AgentId("lead_reviewer", "default")

### Run review

In [10]:
from pathlib import Path

CODE_PATH = Path.cwd() / "sample_review_target.py"

if not CODE_PATH.is_file():
    raise FileNotFoundError(
        f"Expected {CODE_PATH.resolve()} to be present in the current working directory"
    )

SAMPLE_CODE = CODE_PATH.read_text(encoding="utf-8")
print(f"Reviewing: {CODE_PATH.resolve()} ({len(SAMPLE_CODE)} chars)")

response = await worker.send_message(Message(content=SAMPLE_CODE), agent_id)
display(Markdown(response.content))

Reviewing: /Users/damolaadewunmi/Desktop/Work/Andela/agents/5_autogen/community_contributions/code_reviewer/sample_review_target.py (266 chars)


## Style review
- **Readability**: 
  - Use consistent naming conventions; consider using `snake_case` for function names (e.g., `load_user_prefs` is good, `run_report` is acceptable).
  
- **Maintainability**:
  - **Path Construction**: Consider moving the path construction logic to a separate constant or function to enhance ease of updates.
  - **Error Handling**: Introduce error handling for file operations and for the subprocess call to ensure robustness.
  - **String Formatting**: Use `os.path.join` for path construction instead of f-strings to improve platform independence. 

- **Constants**: 
  - Define constants for repeated values, such as the file extension (`.pkl`) and the data directory (`/data/`). 

- **Input Handling**: 
  - Validate or sanitize user input before using it in shell commands to enhance maintainability and future-proofing against potential changes in requirements.

## Security review
- **Pickle use**: High - Using `pickle` for untrusted input can lead to arbitrary code execution if the file is tampered with.
- **File path construction**: Medium - Using user input (`user_id`) to construct a file path without validation can lead to path traversal attacks.
- **OS command execution**: High - The use of `os.system` with user input (`q`) makes the code vulnerable to command injection. An attacker could manipulate `q` to execute arbitrary commands.
- **No input sanitization**: Medium - Lack of input validation for both `user_id` and `q` can lead to various injection attacks and security issues.

## Lead summary

(1) **Executive Summary**: 
The specialist reviews highlight significant issues in style and security within the code under review. Key areas of concern include inconsistent naming conventions, insufficient input validation, and vulnerabilities associated with the use of `pickle` and user input handling. The recommendations aim to enhance code readability, maintainability, and security.

(2) **Merged Findings by Theme**:
- **Readability and Maintainability**: 
  - Consistent naming conventions (use of `snake_case` for functions) is recommended.
  - Path construction logic should be centralized for easier updates.
  - Implement error handling for file operations and subprocess calls.
  - Prefer `os.path.join` over f-strings for platform independence.
  - Constants should be defined for repeated values like file extensions and directories.
  
- **Security**:
  - High risk associated with `pickle` usage for untrusted input due to potential arbitrary code execution.
  - Medium risk involving file path construction from unvalidated user inputs, which could lead to path traversal vulnerabilities.
  - High risk presented by `os.system` usage for command execution with unvalidated user input, exposing the code to command injection risks.
  - Lack of input sanitization poses various security concerns, making the code vulnerable to injection attacks.

(3) **Recommendation**: Request changes. Address inconsistencies in naming conventions, introduce error handling, validate user inputs, and mitigate security vulnerabilities, particularly concerning the use of `pickle` and command execution.

### Shutdown


In [ ]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await w_style.stop()
    await w_sec.stop()

await host.stop()